# Deepfake Detection Prediction - Local Development
Predict whether a video or image is REAL or FAKE using the trained ViT model.

In [ ]:
import os
import sys
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from pathlib import Path

# Add backend to path
BASE_DIR = Path(os.getcwd()).parents[1]
sys.path.append(str(BASE_DIR / "backend"))

from vit_model import ViTDeepfakeDetector, predict_with_vit

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load the model
model = ViTDeepfakeDetector(
    img_size=224,
    patch_size=16,
    embed_dim=192,
    depth=3,
    num_heads=3
).to(device)

MODEL_PATH = "../../backend/model.pth"
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()
    print(f"Loaded model weights from {MODEL_PATH}")
else:
    print(f"WARNING: Model weights not found at {MODEL_PATH}")

In [ ]:
def run_prediction(video_path):
    print(f"Analyzing: {video_path}")
    
    # Extract frames
    cap = cv2.VideoCapture(video_path)
    frames = []
    while len(frames) < 10:
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    cap.release()
    
    if not frames:
        print("Error: Could not read frames from video.")
        return
        
    # Use the same prediction logic as the backend
    result = predict_with_vit(model, frames)
    
    label = "FAKE" if result['prediction'] == 1 else "REAL"
    confidence = result['confidence'] * 100
    
    print(f"Prediction: {label} ({confidence:.2f}%)")
    
    # Show the last frame
    plt.imshow(frames[-1])
    plt.title(f"{label} ({confidence:.2f}%)")
    plt.axis('off')
    plt.show()

In [ ]:
# Test on a file if it exists
test_file = "datasets/real_and_fake_face/training_fake/easy_100_1111.jpg"
if os.path.exists(test_file):
    run_prediction(test_file)
else:
    print("Update 'test_file' path to a valid file to test prediction.")